# Audition the synthetic speaker-profile test corpus

Listen to the 40 clips written by `scripts/gen_synthetic_test_audio.py` and verify:
- The three deliberate speakers (A=`slt`, B=`ksp`, C=`clb`) sound clearly distinct.
- Long passages are intelligible.
- DDK approximation is honest about not being clinically authentic.
- PII clips render digits / symbols the way you'd expect for ASR-then-PII workflows.
- Edge-case sub-1 s clips are usable.

All clips are under `src/tests/data_for_testing/synthetic/`. The manifest documents speaker_id, transcript, duration, and any clinical/PII labels.

In [1]:
import json
from pathlib import Path

from IPython.display import Audio, Markdown, display

# Resolve the corpus relative to this notebook (notebooks/).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SYNTHETIC = REPO_ROOT / "src" / "tests" / "data_for_testing" / "synthetic"
MANIFEST = json.loads((SYNTHETIC / "manifest.json").read_text())
CLIPS = {c["file_id"]: c for c in MANIFEST["clips"]}

print(f"corpus dir: {SYNTHETIC}")
print(f"manifest:   {len(CLIPS)} clips, seed={MANIFEST['seed']}, sr={MANIFEST['sample_rate_hz']} Hz")
print("speakers:")
for k, info in MANIFEST["speakers"].items():
    obs = info["observed_cmu_arctic_speaker_id"]
    exp = info["expected_cmu_arctic_speaker_id"]
    marker = "OK" if obs == exp else "MISMATCH"
    print(f"  {k}: idx={info['xvector_index']:>5}  expected={exp!r:<6} observed={obs!r:<6} [{marker}]")

corpus dir: /orcd/home/002/wilke18/senselab/src/tests/data_for_testing/synthetic
manifest:   40 clips, seed=20260527, sr=16000 Hz
speakers:
  A: idx= 7306  expected='slt'  observed='slt'  [OK]
  B: idx= 5306  expected='ksp'  observed='ksp'  [OK]
  C: idx= 2837  expected='clb'  observed='clb'  [OK]


In [2]:
def play_clip(file_id: str) -> None:
    """Show the transcript + an inline audio player for one clip."""
    meta = CLIPS.get(file_id)
    if meta is None:
        display(Markdown(f"**MISSING** `{file_id}`"))
        return
    speaker = f"{meta['speaker_id']} / `{meta['speaker_id_cmu_arctic']}`"
    label = meta.get("clinical_task") or meta.get("pii_category") or "default"
    transcript = meta.get("transcript", "")
    notes = meta.get("notes") or {}
    note_md = f"  \n_Note: {next(iter(notes.values()))}_" if notes else ""
    header = (
        f"**`{file_id}`** &nbsp; · &nbsp; {meta['duration_s']:.2f}s &nbsp; · &nbsp; "
        f"speaker {speaker} &nbsp; · &nbsp; *{label}*  \n"
        f"> {transcript}{note_md}"
    )
    display(Markdown(header))
    display(Audio(filename=str(SYNTHETIC / file_id)))


def play_clips(file_ids: list[str]) -> None:
    """Play a list of clips in order with a thin separator."""
    for fid in file_ids:
        play_clip(fid)
        display(Markdown("---"))

## 1 — Speaker contrast (most important check)

A↔B should be obviously different (US female vs Indian male).  
A↔C should sound like *same gender, different person* (two US females).

In [3]:
play_clips(
    [
        "sub-A-confident/ses-1/harvard-00.flac",  # A (slt)
        "speaker-B/clip-00.flac",  # B (ksp)
        "speaker-C/clip-00.flac",  # C (clb)
    ]
)

**`sub-A-confident/ses-1/harvard-00.flac`** &nbsp; · &nbsp; 2.62s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *default*  
> The birch canoe slid on the smooth planks.

---

**`speaker-B/clip-00.flac`** &nbsp; · &nbsp; 2.82s &nbsp; · &nbsp; speaker B / `ksp` &nbsp; · &nbsp; *default*  
> The hogs were fed chopped corn and garbage.

---

**`speaker-C/clip-00.flac`** &nbsp; · &nbsp; 2.98s &nbsp; · &nbsp; speaker C / `clb` &nbsp; · &nbsp; *default*  
> The box was thrown beside the parked truck.

---

## 2 — Long passages (intelligibility / quality)

In [4]:
play_clips(
    [
        "sub-A-confident/ses-1/rainbow.flac",
        "sub-A-confident/ses-1/north-wind.flac",
        "sub-A-confident/ses-1/grandfather.flac",
    ]
)

**`sub-A-confident/ses-1/rainbow.flac`** &nbsp; · &nbsp; 18.53s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *rainbow-passage*  
> When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. The rainbow is a division of white light into many beautiful colors. These take the shape of a long round arch, with its path high above, and its two ends apparently beyond the horizon.

---

**`sub-A-confident/ses-1/north-wind.flac`** &nbsp; · &nbsp; 14.72s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *north-wind-and-the-sun*  
> The North Wind and the Sun were disputing which was the stronger, when a traveller came along wrapped in a warm cloak. They agreed that the one who first succeeded in making the traveller take his cloak off should be considered stronger than the other.

---

**`sub-A-confident/ses-1/grandfather.flac`** &nbsp; · &nbsp; 17.41s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *grandfather-passage*  
> You wished to know all about my grandfather. Well, he is nearly ninety-three years old. He dresses himself in an ancient black frock coat, usually minus several buttons. A long beard clings to his chin, giving those who observe him a pronounced feeling of the utmost respect.

---

## 3 — DDK approximation (verify it's labeled honestly)

Expected: SpeechT5 will read this with natural prosody between syllables, **not** the rapid rhythmic articulation of a real clinical DDK. The manifest marks this clip as `tts_approximation: true`.

In [5]:
play_clip("sub-A-confident/ses-1/ddk-pataka-approx.flac")

**`sub-A-confident/ses-1/ddk-pataka-approx.flac`** &nbsp; · &nbsp; 5.15s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *ddk-pataka*  
> Pa-ta-ka, pa-ta-ka, pa-ta-ka, pa-ta-ka, pa-ta-ka, pa-ta-ka.  
_Note: SpeechT5 reads this with natural prosody between syllables rather than the rhythmic rapid articulation of clinical DDK; useful as a repeated-nonsense-syllable stress test only._

## 4 — PII (listen for how digits and symbols come out)

All identifiers are fictional. TTS will pronounce digits as words ("five five five") and `@` / `.` as "at" / "dot" — useful intel for what an ASR-then-PII pipeline will see.

In [10]:
pii_files = sorted(fid for fid in CLIPS if fid.startswith("pii-test/"))
play_clips(pii_files)

**`pii-test/address-spoken.flac`** &nbsp; · &nbsp; 3.55s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *ADDRESS*  
> I live at twelve thirty-four Maple Street in Springfield.

---

**`pii-test/contact-phone.flac`** &nbsp; · &nbsp; 4.58s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *PHONE*  
> You can reach me at five five five, one two three, four five six seven.

---

**`pii-test/dob-spoken.flac`** &nbsp; · &nbsp; 2.85s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *DOB*  
> I was born on March third, nineteen eighty-five.

...

---

**`pii-test/intro-name.flac`** &nbsp; · &nbsp; 1.89s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *NAME*  
> Hi, my name is John Smith.

---

**`pii-test/medication.flac`** &nbsp; · &nbsp; 2.43s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *MEDICATION*  
> I take ibuprofen for my headaches.

---

## 5 — Edge cases (short / sub-1 s clips)

In [7]:
short_files = [
    "sub-A-insufficient/ses-1/single-word.flac",  # 0.51 s
    *sorted(fid for fid in CLIPS if "/animal-naming-" in fid),
    *sorted(fid for fid in CLIPS if "/picture-naming-" in fid),
    "sub-A-confident/ses-1/counting.flac",
]
play_clips(short_files)

**`sub-A-insufficient/ses-1/single-word.flac`** &nbsp; · &nbsp; 0.51s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *default*  
> Rice.

---

**`sub-A-confident/ses-1/animal-naming-00.flac`** &nbsp; · &nbsp; 0.42s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *animal-naming*  
> Cat.

---

**`sub-A-confident/ses-1/animal-naming-01.flac`** &nbsp; · &nbsp; 0.54s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *animal-naming*  
> Dog.

...

---

**`sub-A-confident/ses-1/picture-naming-04.flac`** &nbsp; · &nbsp; 0.77s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *picture-naming*  
> Telephone.

---

**`sub-A-confident/ses-1/counting.flac`** &nbsp; · &nbsp; 3.68s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *counting-1-10*  
> One, two, three, four, five, six, seven, eight, nine, ten.

---

## 6 — Everything else (Harvard sentences + thin subject + intruder)

Skip if you've heard enough; included for completeness.

In [8]:
played = set()
for section in (
    "#### sub-A-confident — Harvard sentences",
    "#### sub-A-thin",
    "#### speaker-B (extra clips)",
):
    pass

groups = {
    "sub-A-confident — Harvard sentences": [f for f in sorted(CLIPS) if f.startswith("sub-A-confident/ses-1/harvard-")],
    "sub-A-thin": [f for f in sorted(CLIPS) if f.startswith("sub-A-thin/")],
    "speaker-B (extra clips)": [f for f in sorted(CLIPS) if f.startswith("speaker-B/")],
}
for header, files in groups.items():
    display(Markdown(f"#### {header}"))
    play_clips(files)

#### sub-A-confident — Harvard sentences

**`sub-A-confident/ses-1/harvard-00.flac`** &nbsp; · &nbsp; 2.62s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *default*  
> The birch canoe slid on the smooth planks.

---

**`sub-A-confident/ses-1/harvard-01.flac`** &nbsp; · &nbsp; 2.37s &nbsp; · &nbsp; speaker A / `slt` &nbsp; · &nbsp; *default*  
> Glue the sheet to the dark blue background.

---

...

---

**`speaker-B/clip-01.flac`** &nbsp; · &nbsp; 2.50s &nbsp; · &nbsp; speaker B / `ksp` &nbsp; · &nbsp; *default*  
> Four hours of steady work faced us.

---

**`speaker-B/clip-02.flac`** &nbsp; · &nbsp; 3.30s &nbsp; · &nbsp; speaker B / `ksp` &nbsp; · &nbsp; *default*  
> A large size in stockings is hard to sell.

---

## Cherry-pick any clip

Use `play_clip("<file_id>")` to listen to one specifically, or list everything:

In [9]:
for fid in sorted(CLIPS):
    meta = CLIPS[fid]
    label = meta.get("clinical_task") or meta.get("pii_category") or "default"
    print(f"{meta['duration_s']:5.2f}s  {meta['speaker_id']}  {label:<25s}  {fid}")

 3.55s  A  ADDRESS                    pii-test/address-spoken.flac
 4.58s  A  PHONE                      pii-test/contact-phone.flac
 2.85s  A  DOB                        pii-test/dob-spoken.flac
 3.68s  A  EMAIL                      pii-test/email-spoken.flac
 8.03s  A  MIXED                      pii-test/free-mixed.flac
 5.92s  A  NONE                       pii-test/free-no-pii.flac
 1.89s  A  NAME                       pii-test/intro-name.flac
 2.43s  A  MEDICATION                 pii-test/medication.flac
 2.82s  B  default                    speaker-B/clip-00.flac
 2.50s  B  default                    speaker-B/clip-01.flac
 3.30s  B  default                    speaker-B/clip-02.flac
 2.98s  C  default                    speaker-C/clip-00.flac
 0.42s  A  animal-naming              sub-A-confident/ses-1/animal-naming-00.flac
 0.54s  A  animal-naming              sub-A-confident/ses-1/animal-naming-01.flac
 0.80s  A  animal-naming              sub-A-confident/ses-1/animal-naming-02.f